# Abliterated model comparison

Qwen 2.5 7B Instruct against huihui-ai's abliterated-v2 derivative. Writes `data/raw/abliterated_queries.csv`, `abliterated_trajectories.csv` and `abliterated_summary.csv`.

In [ ]:
import os

# before importing vllm, see README
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "FLASH_ATTN")
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
os.environ.setdefault("VLLM_DISABLE_FLASHINFER", "1")
os.environ.setdefault("FLASHINFER_DISABLE_VERSION_CHECK", "1")

import json
import random
import re
import string
import time
from pathlib import Path

import numpy as np
import pandas as pd
from vllm import LLM, SamplingParams

In [ ]:
SEED = 7

MODELS = {
    "qwen25_7b_parent": ("Qwen/Qwen2.5-7B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28"),
    "qwen25_7b_abliterated_v2": ("huihui-ai/Qwen2.5-7B-Instruct-abliterated-v2", "447ff10df7c9b7031f28eec54b9042a362d09696"),
}
ACTIVE = ["qwen25_7b_parent"]       # one model per session
LLM_KWARGS = dict(trust_remote_code=True, tensor_parallel_size=1, gpu_memory_utilization=0.85,
                  max_model_len=4096, max_num_seqs=12, enforce_eager=True)
BATCH_SIZE = 12

TEMPERATURE = 0.2
TOP_P = 0.9
NAME_LENGTH = 3
IDEOLOGY = ["liberalism", "conservatism", "socialism", "libertarianism", "green politics", "nationalism"]

# responses: (block, labels, q, N, queries per state); labels None = neutra
BLOCKS = [
    ("neutral_q2_N100", None, 2, 100, 150),
    ("neutral_q10_N100", None, 10, 100, 150),
    ("neutral_q10_N200", None, 10, 200, 150),
    ("neutral_q50_N200", None, 50, 200, 150),
    ("neutral_matched_political_q6_N50", None, 6, 50, 240),
    ("political_ideology_q6_N50", IDEOLOGY, 6, 50, 240),
]
REL_GRID = [0.0, 0.1, 0.25, 0.4, 0.6, 0.8, 1.0]
SMALL_LEADS = [0.03, 0.06, 0.15, 0.20]
MAX_NEW_TOKENS_RESPONSE = 24

# dynamics: (condition, labels, q, N)
CONDITIONS = [
    ("neutral_q2_N100", None, 2, 100),
    ("neutral_q10_N100", None, 10, 100),
    ("neutral_q10_N200", None, 10, 200),
    ("neutral_q50_N200", None, 50, 200),
    ("political_ideology_q6_N50", IDEOLOGY, 6, 50),
    ("neutral_matched_political_q6_N50", None, 6, 50),
]
N_RUNS = 3
T_MAX_SWEEPS = 100
RECORDS_PER_SWEEP = 10
MAX_NEW_TOKENS_DYNAMICS = 16
SEED_OFFSET = 90_000_000

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## Prompt and parsers

In [ ]:
def neutral_labels(q):
    return [a + b for a in string.ascii_lowercase for b in string.ascii_lowercase][:q]


def random_names(n, rng):
    chars = string.ascii_letters + string.digits
    names = set()
    while len(names) < n:
        names.add("".join(rng.choices(chars, k=NAME_LENGTH)))
    return list(names)


def create_prompt(names, opinions):
    lines = ["Below you can see the list of all the other AI agents with the opinion they support.",
             "You must reply with the opinion you want to support.",
             "The opinion must be reported between square brackets.", ""]
    lines += [f"{n}: {o}" for n, o in zip(names, opinions)]
    lines.append("Reply only with the opinion you want to support, between square brackets.")
    return "\n".join(lines)


def parse_reply(text, labels):
    # controlled measurement: first bracket content that is a label, case-insensitive
    found = re.findall(r"\[([^\]]+)\]", text)
    lowered = {l.lower(): l for l in labels}
    for c in found if found else [text]:
        c = c.strip().strip(" .,:;!?'\"").lower()
        if c in lowered:
            return lowered[c]
    return None


def parse_reply_dynamics(text, labels):
    # dynamics: exact label in brackets, otherwise a reply that contains exactly one label
    found = re.findall(r"\[([^\]]+)\]", text)
    exact = {c.strip().strip(" .,:;!?'").strip('"') for c in (found if found else [text])} & set(labels)
    if len(exact) == 1:
        return exact.pop()
    if len(exact) > 1:
        return None
    contained = [l for l in labels if re.search(rf"(?<!\w){re.escape(l)}(?!\w)", text.strip())]
    return contained[0] if len(contained) == 1 else None


def chat_format(tok, prompts):
    return [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

## Controlled responses

In [ ]:
def leader_counts(q, n_display, rel):
    c_min = n_display // q + (1 if n_display % q else 0)
    c_max = n_display - (q - 1)
    leader = round(c_min + rel * (c_max - c_min))
    rest = n_display - leader
    return [leader] + [rest // (q - 1) + (1 if i < rest % (q - 1) else 0) for i in range(q - 1)]


def design_points(q, n_display):
    rels = REL_GRID + (SMALL_LEADS if q >= 3 else [])
    points, seen = [], set()
    for rel in sorted(rels):
        counts = leader_counts(q, n_display, rel)
        if tuple(counts) not in seen:
            seen.add(tuple(counts))
            points.append((rel, counts))
    return points


def build_query(counts, labels, i, rng):
    # neutral: fresh permutation; ideology: label (i + role) % q takes role `role`, so every label
    # sits in every count slot once per six queries
    q = len(labels)
    if labels == IDEOLOGY:
        display = [labels[(i + role) % q] for role in range(q)]
    else:
        display = list(labels)
        rng.shuffle(display)
    opinions = []
    for role, c in enumerate(counts):
        opinions += [display[role]] * c
    names = random_names(len(opinions), rng)
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    return create_prompt([n for n, _ in pairs], [o for _, o in pairs]), {d: r for r, d in enumerate(display)}


def run_state(llm, tok, model_label, block, labels, q, N, rel, counts, n_queries, seed):
    rng = random.Random(seed)
    queries = [build_query(counts, labels, i, rng) for i in range(n_queries)]
    rows = []
    for start in range(0, n_queries, BATCH_SIZE):
        batch = queries[start:start + BATCH_SIZE]
        outputs = llm.generate(chat_format(tok, [b[0] for b in batch]), sampling, use_tqdm=False)
        for (_, display_to_role), out in zip(batch, outputs):
            chosen = parse_reply(out.outputs[0].text, list(display_to_role))
            leader = next(d for d, r in display_to_role.items() if r == 0)
            rows.append({"model_label": model_label, "block_id": block,
                         "label_mode": "semantic" if labels == IDEOLOGY else "neutral_per_query",
                         "q": q, "N": N, "n_display": N - 1, "rel": rel, "leader_share": counts[0] / (N - 1),
                         "counts_role": json.dumps(counts), "designated_leader_label": leader,
                         "chosen_display": chosen, "chosen_role": display_to_role.get(chosen) if chosen else None,
                         "reply": out.outputs[0].text})
    return pd.DataFrame(rows)


sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS_RESPONSE)
out_file = RAW / "abliterated_queries.csv"
cell = 0
for model_label in ACTIVE:
    model_id, revision = MODELS[model_label]
    llm = LLM(model=model_id, revision=revision, disable_log_stats=True, **LLM_KWARGS)
    tok = llm.get_tokenizer()
    for block, labels, q, N, n_queries in BLOCKS:
        for rel, counts in design_points(q, N - 1):
            cell += 1
            t0 = time.time()
            df = run_state(llm, tok, model_label, block, labels or neutral_labels(q), q, N, rel, counts, n_queries, SEED + cell)
            df.to_csv(out_file, mode="a", header=not out_file.exists(), index=False)
            print(f"{model_label} {block} rel={rel:.2f}  valid={df['chosen_role'].notna().mean():.3f}  "
                  f"P(leader)={(df['chosen_role'] == 0).mean():.3f}  {time.time() - t0:.0f}s")
    del llm

## Opinion dynamics

In [ ]:
def run_seed(q, N, run):
    return SEED + SEED_OFFSET + 10_000 * q + N + 100_000 * run


def initial_counts(q, N, run):
    # balanced startt
    # the same for both models and for the ideology
    base, extra = divmod(N, q)
    order = list(range(q))
    random.Random(SEED + 31_415_927 + 10_000 * q + N).shuffle(order)
    plus_one = [order[(run * extra + j) % q] for j in range(extra)]
    return [base + (1 if i in plus_one else 0) for i in range(q)]


def new_run(model_label, condition, labels, q, N, run):
    labels = list(labels) if labels else neutral_labels(q)
    counts = dict(zip(labels, initial_counts(q, N, run)))
    return {"model_label": model_label, "condition_id": condition, "semantic": labels == IDEOLOGY,
            "q": q, "N": N, "run": run, "seed": run_seed(q, N, run),
            "rng": random.Random(run_seed(q, N, run)), "label_rng": random.Random(run_seed(q, N, run) + 1_000_000_007),
            "labels": labels, "counts": counts, "initial_counts": dict(counts),
            "invalid": 0, "valid": 0, "flips": 0, "reactivations": 0, "consensus_step": None}


def build_prompt(run):
    N, counts, rng, labels = run["N"], run["counts"], run["rng"], run["labels"]
    shown = labels[:]
    if not run["semantic"]:
        run["label_rng"].shuffle(shown)          # fresh label permutation, ideologies stay as they are here
    to_display = dict(zip(labels, shown))
    to_internal = dict(zip(shown, labels))
    names = random_names(N, rng)
    opinions = [l for l in labels for _ in range(counts[l])]
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    i = rng.randint(0, N - 1)                    # focal agent, not shown in the list
    others = [(n, to_display[o]) for k, (n, o) in enumerate(pairs) if k != i]
    return create_prompt([n for n, _ in others], [o for _, o in others]), pairs[i][1], to_internal


def apply_reply(run, own, to_internal, text):
    chosen = parse_reply_dynamics(text, list(to_internal))    # extinct labels stay valid (potts)
    if chosen is None:
        run["invalid"] += 1
        return
    run["valid"] += 1
    chosen = to_internal[chosen]
    if chosen == own:
        return
    if run["counts"][chosen] == 0:
        run["reactivations"] += 1
    run["counts"][own] -= 1
    run["counts"][chosen] += 1
    run["flips"] += 1


def record(run, step, record_type):
    counts, N = run["counts"], run["N"]
    return {"model_label": run["model_label"], "condition_id": run["condition_id"], "q": run["q"], "N": N,
            "run": run["run"], "step": step, "time_sweeps": step / N, "record_type": record_type,
            "counts": json.dumps(counts), "leader_share": max(counts.values()) / N,
            "n_active_opinions": sum(c > 0 for c in counts.values()), "invalid_count": run["invalid"]}


def summary(run, model_id, revision):
    counts, N, q = run["counts"], run["N"], run["q"]
    done = run["consensus_step"] is not None
    top = max(counts.values())
    return {"model_label": run["model_label"], "model_id": model_id, "model_revision": revision,
            "condition_id": run["condition_id"], "label_mode": "semantic" if run["semantic"] else "neutral_per_query",
            "semantic_set": "political_ideology" if run["semantic"] else "", "q": q, "N": N, "run": run["run"],
            "run_seed": run["seed"], "initial_counts": json.dumps(run["initial_counts"]),
            "scheduled_horizon_sweeps": T_MAX_SWEEPS, "event_observed": done,
            "consensus_step": run["consensus_step"] if done else np.nan,
            "consensus_time_sweeps": run["consensus_step"] / N if done else np.nan,
            "observed_time_sweeps": run["consensus_step"] / N if done else T_MAX_SWEEPS,
            "counts": json.dumps(counts), "leader_share": top / N,
            "n_active_opinions": sum(c > 0 for c in counts.values()),
            "winner_label": "|".join(sorted(l for l, c in counts.items() if c == top)),
            "surviving_opinions": json.dumps([l for l, c in counts.items() if c > 0]),
            "invalid_count": run["invalid"], "valid_count": run["valid"], "flip_count": run["flips"],
            "reactivation_count": run["reactivations"],
            "valid_response_rate": run["valid"] / max(1, run["valid"] + run["invalid"]),
            "temperature": TEMPERATURE, "top_p": TOP_P, "max_new_tokens": MAX_NEW_TOKENS_DYNAMICS}

In [ ]:
def run_condition(llm, tok, model_label, model_id, revision, condition, labels, q, N):
    runs = [new_run(model_label, condition, labels, q, N, r) for r in range(N_RUNS)]
    t_max = T_MAX_SWEEPS * N
    every = max(1, N // RECORDS_PER_SWEEP)
    rows = [record(run, 0, "regular") for run in runs]
    t0 = time.time()
    for step in range(1, t_max + 1):
        active = [run for run in runs if run["consensus_step"] is None]     # consensus is absorbing
        prompts = [build_prompt(run) for run in active]
        for start in range(0, len(active), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            outputs = llm.generate(chat_format(tok, [p[0] for p in batch]), sampling, use_tqdm=False)
            for run, (_, own, to_internal), out in zip(active[start:start + BATCH_SIZE], batch, outputs):
                apply_reply(run, own, to_internal, out.outputs[0].text)
        for run in active:
            if max(run["counts"].values()) == N:
                run["consensus_step"] = step
                if step % every:
                    rows.append(record(run, step, "consensus_event"))
        if step % every == 0 or step == t_max:
            rows += [record(run, step, "regular") for run in runs]
        if step % N == 0:
            print(f"{model_label} {condition}  t={step // N}/{T_MAX_SWEEPS}  "
                  f"consensus={sum(r['consensus_step'] is not None for r in runs)}/{N_RUNS}  {time.time() - t0:.0f}s")
    return pd.DataFrame(rows), pd.DataFrame([summary(run, model_id, revision) for run in runs])


sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS_DYNAMICS)
for model_label in ACTIVE:
    model_id, revision = MODELS[model_label]
    llm = LLM(model=model_id, revision=revision, disable_log_stats=True, **LLM_KWARGS)
    tok = llm.get_tokenizer()
    for condition, labels, q, N in CONDITIONS:
        traj, summ = run_condition(llm, tok, model_label, model_id, revision, condition, labels, q, N)
        for name, df in [("abliterated_trajectories.csv", traj), ("abliterated_summary.csv", summ)]:
            df.to_csv(RAW / name, mode="a", header=not (RAW / name).exists(), index=False)
        print(summ[["condition_id", "run", "event_observed", "consensus_time_sweeps", "winner_label"]].to_string(index=False))
    del llm